In [ ]:
%matplotlib inline

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "groq", "httpx==0.27.0"],
               capture_output=True)
print("✓ Libraries ready")

In [ ]:
from groq import Groq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime

print("✓ All imports successful")
print(f"  Pandas  : {pd.__version__}")
print(f"  NumPy   : {np.__version__}")

In [ ]:
from groq import Groq
from fpdf import FPDF
import json
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

GROQ_KEY = input("Enter your Groq API key: ")
def save_pdf(report, filename, df):
    pdf = FPDF()
    pdf.add_page()

    # HEADER
    pdf.set_fill_color(30, 30, 50)
    pdf.rect(0, 0, 210, 40, 'F')
    pdf.set_font("Helvetica", 'B', 20)
    pdf.set_text_color(0, 212, 170)
    pdf.set_y(10)
    pdf.cell(0, 10, "AI EDA REPORT", ln=True, align='C')
    pdf.set_font("Helvetica", '', 11)
    pdf.set_text_color(180, 180, 200)
    pdf.cell(0, 8, f"Dataset: {filename}", ln=True, align='C')

    # META
    pdf.set_y(45)
    pdf.set_font("Helvetica", '', 10)
    pdf.set_text_color(100, 100, 100)
    pdf.cell(0, 6, f"Generated : {datetime.now().strftime('%Y-%m-%d %H:%M')}", ln=True)
    pdf.cell(0, 6, f"Rows      : {df.shape[0]:,}   |   Columns : {df.shape[1]}", ln=True)
    pdf.ln(5)

    # DIVIDER
    pdf.set_draw_color(0, 212, 170)
    pdf.set_line_width(0.8)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(6)

    # BODY
    pdf.set_font("Helvetica", '', 11)
    pdf.set_text_color(30, 30, 30)

   # Clean special characters Helvetica can't handle
    report = report.replace('—', '-').replace('–', '-').replace('"', '"').replace('"', '"').replace("'", "'").replace("'", "'").replace('•', '-').replace('…', '...')

    for line in report.split('\n'):
        line = line.strip()
        if not line:
            pdf.ln(3)
            continue
        if line.startswith('##') or (line.isupper() and len(line) > 4):
            pdf.ln(4)
            pdf.set_font("Helvetica", 'B', 13)
            pdf.set_text_color(0, 100, 140)
            clean = line.replace('##', '').replace('**', '').strip()
            pdf.cell(0, 8, clean, ln=True)
            pdf.set_draw_color(200, 220, 230)
            pdf.set_line_width(0.3)
            pdf.line(10, pdf.get_y(), 200, pdf.get_y())
            pdf.ln(3)
            pdf.set_font("Helvetica", '', 11)
            pdf.set_text_color(30, 30, 30)
        elif line.startswith('-') or line.startswith('*') or line.startswith('•'):
            clean = line.lstrip('-*• ').replace('**', '').strip()
            pdf.set_x(15)
            pdf.set_font("Helvetica", '', 10)
            pdf.set_text_color(50, 50, 50)
            pdf.cell(5, 6, "-")
            pdf.set_x(20)
            pdf.multi_cell(175, 6, clean)
            pdf.set_font("Helvetica", '', 11)
            pdf.set_text_color(30, 30, 30)
        else:
            clean = line.replace('**', '').strip()
            pdf.multi_cell(0, 6, clean)

    # FOOTER
    pdf.set_y(-20)
    pdf.set_font("Helvetica", 'I', 8)
    pdf.set_text_color(150, 150, 150)
    pdf.cell(0, 6, "Generated by AI EDA Reporter - Python + LLaMA 3", align='C')
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    out_file = f"eda_report_{filename}_{timestamp}.pdf"
    pdf.output(out_file)
    print(f"✓ PDF saved as: {out_file}")
    return out_file


def analyze(df, filename="dataset"):
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()

    # 1. BASIC PROFILE
    print("=" * 55)
    print("  DATASET PROFILE")
    print("=" * 55)
    print(f"  Rows             : {df.shape[0]:,}")
    print(f"  Columns          : {df.shape[1]}")
    print(f"  Numeric cols     : {len(num_cols)}")
    print(f"  Categorical cols : {len(cat_cols)}")
    print(f"  Duplicate rows   : {df.duplicated().sum()}")
    print(f"  Total missing    : {df.isnull().sum().sum()}")
    print()

    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print("  MISSING VALUES:")
        for col, count in missing.items():
            pct = count / len(df) * 100
            bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
            print(f"  {col:30s} {bar} {pct:.1f}%")
    else:
        print("  ✓ No missing values found")
    print()

    # 2. STATISTICS
    if num_cols:
        print("  DESCRIPTIVE STATISTICS:")
        print(df[num_cols].describe().round(2).to_string())
        print()

    # 3. DISTRIBUTION CHARTS
    if num_cols:
        cols_to_plot = num_cols[:6]
        rows = (len(cols_to_plot) + 2) // 3
        fig, axes = plt.subplots(rows, 3, figsize=(15, 4 * rows))
        fig.patch.set_facecolor('#f0f4f8')
        axes_flat = np.array(axes).flatten()
        for i, col in enumerate(cols_to_plot):
            ax = axes_flat[i]
            df[col].dropna().hist(bins=30, ax=ax, color='#2196F3', alpha=0.8, edgecolor='white')
            ax.set_title(col, fontsize=11, fontweight='bold', pad=8)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_facecolor('#ffffff')
        for j in range(i + 1, len(axes_flat)):
            axes_flat[j].set_visible(False)
        plt.suptitle('Distributions', fontsize=14, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.show()

    # 4. BOXPLOTS
    if num_cols:
        cols_to_box = num_cols[:6]
        fig, axes = plt.subplots(1, len(cols_to_box), figsize=(3 * len(cols_to_box), 5))
        if len(cols_to_box) == 1:
            axes = [axes]
        for i, col in enumerate(cols_to_box):
            axes[i].boxplot(df[col].dropna(), patch_artist=True,
                            boxprops=dict(facecolor='#2196F3', alpha=0.6),
                            medianprops=dict(color='#ff6b35', linewidth=2),
                            whiskerprops=dict(color='#555'),
                            capprops=dict(color='#555'))
            axes[i].set_title(col, fontsize=10, fontweight='bold')
            axes[i].spines['top'].set_visible(False)
            axes[i].spines['right'].set_visible(False)
        plt.suptitle('Boxplots — Outlier Detection', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    # 5. CORRELATION HEATMAP
    if len(num_cols) >= 2:
        fig, ax = plt.subplots(figsize=(10, 7))
        corr = df[num_cols].corr()
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                    cmap='coolwarm', center=0, ax=ax,
                    linewidths=0.5, linecolor='white',
                    cbar_kws={'shrink': 0.7})
        ax.set_title('Correlation Matrix', fontsize=13, fontweight='bold', pad=15)
        plt.tight_layout()
        plt.show()

    # 6. CATEGORICAL BAR CHARTS
    if cat_cols:
        cols_to_bar = cat_cols[:4]
        fig, axes = plt.subplots(1, len(cols_to_bar), figsize=(5 * len(cols_to_bar), 5))
        if len(cols_to_bar) == 1:
            axes = [axes]
        colors = ['#2196F3', '#00d4aa', '#ff6b35', '#7c6af7']
        for i, col in enumerate(cols_to_bar):
            top = df[col].value_counts().head(6)
            axes[i].barh(top.index.astype(str), top.values, color=colors[i % len(colors)], alpha=0.8)
            axes[i].set_title(col, fontsize=11, fontweight='bold')
            axes[i].spines['top'].set_visible(False)
            axes[i].spines['right'].set_visible(False)
            axes[i].invert_yaxis()
        plt.suptitle('Top Categories per Column', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    # 7. BUILD JSON PROFILE
    missing_full = df.isnull().sum()
    profile = {
        "filename": filename,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": df.columns.tolist(),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing": {
            col: {"count": int(missing_full[col]), "pct": round(missing_full[col]/len(df)*100, 2)}
            for col in df.columns if missing_full[col] > 0
        },
        "numeric_stats": df[num_cols].describe().round(3).to_dict() if num_cols else {},
        "categorical_stats": {
            col: {
                "unique_count": int(df[col].nunique()),
                "top_values": df[col].value_counts().head(4).to_dict()
            }
            for col in cat_cols[:5]
        },
    }

    if len(num_cols) >= 2:
        corr_matrix = df[num_cols].corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        pairs = upper.stack().sort_values(ascending=False)
        profile["top_correlations"] = [
            {"col1": str(a), "col2": str(b), "correlation": round(float(v), 3)}
            for (a, b), v in pairs.head(5).items()
        ]

    # 8. AI REPORT
    print("=" * 55)
    print("  AI BUSINESS INSIGHT REPORT")
    print("=" * 55)

    client = Groq(api_key=GROQ_KEY)

    prompt = f"""You are a senior data analyst preparing a business insight report.
You have been given an automated profile of a dataset called '{filename}'.

Data profile:
{json.dumps(profile, indent=2, default=str)}

Write a professional report with these 6 sections:
1. DATASET OVERVIEW
2. DATA QUALITY ASSESSMENT
3. KEY STATISTICAL FINDINGS
4. CATEGORICAL INSIGHTS
5. CORRELATION HIGHLIGHTS
6. RECOMMENDED NEXT STEPS

Use bullet points under each section, not paragraphs.
Each bullet must reference actual column names and real numbers from the profile.
Minimum 3 bullets per section. Be specific and analytical.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a senior data analyst writing precise business reports."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4,
        max_tokens=1800
    )

    report = response.choices[0].message.content
    print(report)

    # 9. SAVE AS PDF
    save_pdf(report, filename, df)

In [ ]:
df = pd.read_csv("CrimesOnWomenData.csv")
analyze(df, filename="CrimesOnWomenData")

In [ ]:
import os
print(os.getcwd())